In [ ]:
from pathlib import Path

import pandera.polars as pa
from pandera.typing.polars import Series

import polars as pl


import seaborn as sns

from climate_attitudes.extract.dataset import ClimateAttitudesDataset

ASSETS_DIR = Path("/Users/henry/data/msc_thesis/climate-attitudes")

In [ ]:
data = ClimateAttitudesDataset(ASSETS_DIR)

In [ ]:
dem_items = data.item.filter(pl.col("is_demographic"))
dem_questions = dem_items.select("item_id").join(
    data.question, how="left", on="item_id"
)

### Which items are present in which waves?

In [ ]:
with pl.Config(tbl_rows=100):
    print(
        dem_questions.group_by(("item_id", "item_name"), maintain_order=True).agg(
            "wave"
        )
    )

In [ ]:
dem_questions.filter(pl.col("wave") != 1, ~pl.col("repeating_participants")).group_by(
    "item_name"
).agg("wave")

#### Always asked (only-new-participant waves in brackets)
- State, county, zip
- Gender
- Age
- Race, Latino
- Born in USA **(2,3,4)**
- Marital status **(3)**
- Education
- Employment status
- Health insurance status **(3)**
- Income: Total household income, perception of income sufficiency
- Household: number of people **(2,3,4)**, number of people younger than 18 **(2,3,4)**, speaks second language **(3,4)**
- Residence: urban/rural **(2)**, time resided

Special cases:
- Employment status response schema changed in W2
- Health insurance status response schema changed in W5

#### Not always asked (only-new-participant waves in brackets)

**Employment:** Waves 1,2,3,4 asked participants if they worked in public/private/self-employment (`dem_emp_type`), and about the sectors in which members of their 
household are employed (**Only new participants in W3**; `dem_sector`). Wave 4 additionally asked participants who reported that a member of their household 
worked in relation to oil and gas, if they would consider equivalent/better roles in renewable energy (`dem_sector_Reoffer`). Finally, waves 3 and 4 asked participants
if they had the option of working from home (`dem_wfh`).
<!-- - 1234: `dem_emp_type`, `dem_sector` **(3)**
- 34: `dem_wfh`
- 4: `dem_sector_Reoffer` -->

**Income:** Waves 1,2 asked participants how they expected their income to change over the following 12 months (`dem_income_future`).
  
**Religion:** All waves include a question assessing personal importance of religion (Waves 1, 2, and 3: `dem_relig`; Waves 4 and 5: `dem_relig_scale`). In Wave 3 this question was only presented to new participants.
Wave 4 included two extra questions, assessing individuals' particular religious affiliation/identification (`dem_relig_affil`, `dem_relig_bornagain`).
<!-- - 123: `dem_relig` **(3)**
- 4: `dem_relig_affil`, `dem_relig_bornagain`
- 45: `dem_relig_scale` -->

**Health:** In waves 1,2,3,4 **new participants** were asked about health condition diagnoses.
<!-- - 1234: `dem_healthconditions` **(only new participants)** -->

**Welfare:** Waves 2,3,4,5 asked participants about current unemployment insurance/govt benefit status (Waves 2,3,4: `dem_ui`; Wave 5: `dem_welfare`).
Wave 5 also asked about past insurance/benefits (`dem_welfare_past`).
<!-- - 234: `dem_ui` (receiving unemployment insurance or other govt benefits)
- 5: `dem_welfare`, `dem_welfare_past` -->

**Residence:** Wave 5 asked participants about home-ownership status (`dem_hhown`), whether they were living in/using federally-assisted or HUD housing (`dem_hud`), or if they were 
living on an Indian reservation (`dem_res`).
<!-- - 5: `dem_hhown`, `dem_hud`, `dem_res` -->

In [ ]:
question_waves = (
    dem_questions.group_by(("item_id", "item_name"), maintain_order=True)
    .agg("wave")
    .group_by("wave")
    .agg("item_name")
    .sort(by="wave")
)

for waves, waves_questions in question_waves.iter_rows():
    print(f"Waves: {waves}; Questions")

## Who is taking the survey?

We consider the participants in Wave 1 of the survey only (to avoid inconsistent responses).

In [ ]:
class DemSchema(pa.DataFrameModel):
    dem_stcount_1: Series[str] = pa.Field(coerce=True, nullable=False)
    dem_stcount_2: Series[str] = pa.Field(coerce=True, nullable=False)
    dem_zip: Series[str] = pa.Field(coerce=True, nullable=False)
    dem_educ: Series[int] = pa.Field(in_range=(1, 6), coerce=True, nullable=False)
    dem_male: Series[int] = pa.Field(isin=(0, 1, 77), coerce=True, nullable=False)
    dem_age: Series[int] = pa.Field(gt=0, lt=120, coerce=True, nullable=False)
    # dem_race: pl.List = pa.Field(dtype_kwargs={"inner": pl.Int64}, in_range=(1,6), coerce=True, nullable=False)
    dem_latino: bool = pa.Field(coerce=True, nullable=False)
    dem_income: int = pa.Field(in_range=(1, 6), coerce=True, nullable=False)
    dem_urban: int = pa.Field(in_range=(1, 3), coerce=True, nullable=False)
    dem_peoplehh: int = pa.Field(ge=1, coerce=True, nullable=True)
    dem_peopleunder18: int = pa.Field(ge=0, coerce=True, nullable=True)
    dem_health: int = pa.Field(in_range=(1, 4), coerce=True, nullable=False)
    dem_healthconditions: list[int] = pa.Field(coerce=True, nullable=True)
    dem_healthinsurance: int = pa.Field(
        in_range=(0, 4), coerce=True, nullable=True
    )  # NOTE: Order of response values changed
    dem_marital: int = pa.Field(
        in_range=(1, 7), coerce=True, nullable=True
    )  # NOTE: Broken. Should be 1--6 per codebook.
    dem_timeresid: int = pa.Field(in_range=(1, 3), coerce=True, nullable=False)
    dem_income_percep: int = pa.Field(in_range=(1, 4), coerce=True, nullable=False)
    dem_income_future: int = pa.Field(in_range=(-1, 1), coerce=True, nullable=True)
    dem_US: bool = pa.Field(coerce=True, nullable=True)
    dem_relig: bool = pa.Field(coerce=True, nullable=True)
    dem_relig_scale: int = pa.Field(in_range=(1, 4), coerce=True, nullable=True)
    dem_relig_affil: int = pa.Field(
        in_range=(0, 13), coerce=True, nullable=True
    )  # NOTE: Should be enum, but don't know order
    dem_relig_bornagain: bool = pa.Field(coerce=True, nullable=True)
    dem_emp: int = pa.Field(
        in_range=(1, 10), coerce=True, nullable=False
    )  # NOTE: Option values changed at some point
    dem_emp_type: int = pa.Field(in_range=(1, 3), coerce=True, nullable=True)
    # dem_sector: int = pa.Field(isin=(1,2,3,4,5,6,7,77), coerce=True, nullable=True)   # NOTE: Broken. Need to check all elements of list.
    dem_wfh: bool = pa.Field(coerce=True, nullable=True)
    dem_ui: int = pa.Field(in_range=(1, 3), coerce=True, nullable=True)
    dem_lang: bool = pa.Field(coerce=True, nullable=True)
    dem_hhown: int = pa.Field(isin=(1, 2, 5), coerce=True, nullable=True)
    dem_hud: int = pa.Field(isin=(0, 1, 2, 3, 4, 99), coerce=True, nullable=True)
    dem_res: bool = pa.Field(coerce=True, nullable=True)
    dem_welfare: int = pa.Field(isin=(0, 1, 2), coerce=True, nullable=True)
    dem_welfare_past: int = pa.Field(isin=(0, 1, 2), coerce=True, nullable=True)

In [ ]:
dem_parsers = {
    "dem_age": pl.col("dem_age").cast(pl.Float64).cast(pl.Int64),
    "dem_race": pl.col("dem_race")
    .replace("", None)
    .str.split(",")
    .list.eval(pl.element().cast(pl.Int64, strict=True)),
    "dem_latino": pl.col("dem_latino").cast(pl.Int64),
    "dem_urban": pl.col("dem_urban").cast(pl.Float64).cast(pl.Int64),
    "dem_healthconditions": pl.col("dem_healthconditions")
    .replace("", None)
    .str.split(","),
    "dem_peopleunder18": pl.when(pl.col("dem_peopleunder18").cast(pl.Int64) < 0)
    .then(None)
    .otherwise(pl.col("dem_peopleunder18"))
    .alias("dem_peopleunder18"),
    "dem_US": pl.col("dem_US").cast(pl.Int64).cast(pl.Boolean),
    "dem_relig": pl.col("dem_relig").cast(pl.Int64).cast(pl.Boolean),
    "dem_relig_bornagain": pl.col("dem_relig_bornagain")
    .cast(pl.Int64)
    .cast(pl.Boolean),
    "dem_sector": pl.col("dem_sector")
    .replace("", None)
    .str.split(",")
    .list.eval(pl.element().cast(pl.Int64, strict=True)),
    "dem_wfh": pl.col("dem_wfh").cast(pl.Int64).cast(pl.Boolean),
    "dem_lang": pl.col("dem_lang").cast(pl.Int64).cast(pl.Boolean),
    "dem_res": pl.col("dem_res").cast(pl.Int64).cast(pl.Boolean),
}


def parse_dem_cols(df):
    col_parsers = [
        dem_parsers[dem_col]
        for dem_col in df.select(pl.col(r"^dem_.*$")).columns
        if dem_col in dem_parsers
    ]
    return DemSchema.validate(df.with_columns(*col_parsers))

In [ ]:
# w1_pids = data.wave_participants(1)
dem_data = parse_dem_cols(
    data.question_response.filter(
        pl.col("wave") == 1,
        pl.col("item_name").str.starts_with("dem_"),
    )
    # .join(w1_pids, on="participant_id", how="right")
    .select("participant_id", "wave", "item_name", "response")
    .pivot("item_name", values="response")
    # .select(pl.col("dem_healthconditions").replace("", None).str.split(","))
)

In [ ]:
dem_data

In [ ]:
import polars.selectors as cs

sns.pairplot(
    dem_data.drop(
        "participant_id",
        "wave",
        "dem_stcount_1",
        "dem_stcount_2",
        "dem_zip",
        "dem_ui",
        "dem_race",
        "dem_healthconditions",
        "dem_sector",
        "dem_relig_scale",
        "dem_relig_affil",
        "dem_relig_bornagain",
        "dem_hhown",
        "dem_hud",
        "dem_res",
        "dem_welfare",
        "dem_welfare_past",
        cs.boolean(),
    ).to_pandas()
)